In [1]:
import pandas as pd
import random
from collections import defaultdict, Counter
import re
from nltk.util import bigrams, trigrams

In [2]:
file_path = r"data\train.jsonl"

df = pd.read_json(file_path, lines=True)

df.head()

,id,text,source
0,0,From my grandfather Verus I learned good moral...,meditations
1,1,From the reputation and remembrance of my fath...,meditations
2,2,"From my mother, piety and beneficence, and abs...",meditations
3,3,"From my great-grandfather, not to have frequen...",meditations
4,4,"From my governor, to be neither of the green n...",meditations


In [3]:
text = " ".join(df["text"])

In [4]:
len(text)

1381764

In [5]:
def regex_tokenize(text):
    pattern = r"[A-Za-z0-9]+|[.,!?;:()\"'-]"
    return re.findall(pattern, text)

In [6]:
tokens = regex_tokenize(text)

len(tokens)

291981

In [7]:
Counter(tokens).most_common(30)

[(',', 17416),
 ('the', 11401),
 ('.', 8623),
 ('to', 7730),
 ('and', 7279),
 ('of', 7060),
 ('is', 6248),
 ('a', 4655),
 ('that', 4151),
 (';', 3760),
 ('in', 3735),
 ('it', 3387),
 ('not', 2959),
 ('you', 2902),
 ('which', 2662),
 ('be', 2450),
 ('I', 2350),
 ('are', 2291),
 ('"', 2234),
 ('for', 2169),
 ('as', 2039),
 ('he', 1964),
 ('with', 1749),
 ('by', 1743),
 ('have', 1676),
 ('this', 1590),
 ('but', 1508),
 ('man', 1507),
 ('or', 1489),
 ('?', 1489)]

In [8]:
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


In [9]:
trigram = get_ngrams(tokens, 3)

trigram_frequencies = Counter(trigram).most_common(30)

trigram_frequencies

[(('that', 'which', 'is'), 222),
 ((',', 'however', ','), 219),
 ((',', 'and', 'the'), 197),
 (('.', 'It', 'is'), 170),
 (('man', "'", 's'), 141),
 ((',', 'and', 'that'), 129),
 ((',', 'it', 'is'), 122),
 ((',', 'and', 'to'), 117),
 (('the', 'things', 'which'), 112),
 ((';', 'it', 'is'), 108),
 (('.', 'Farewell', '.'), 105),
 ((',', 'then', ','), 104),
 (('the', 'wise', 'man'), 103),
 (('it', 'is', 'not'), 95),
 (('there', 'is', 'no'), 91),
 (('one', "'", 's'), 81),
 (('.', '"', 'But'), 81),
 ((',', 'and', 'not'), 79),
 (('that', 'it', 'is'), 79),
 (('it', 'is', 'a'), 77),
 ((',', 'if', 'you'), 75),
 (('I', 'do', 'not'), 72),
 (('.', 'There', 'is'), 70),
 (('.', 'Do', 'you'), 70),
 (('.', 'Let', 'us'), 68),
 ((',', '"', 'you'), 68),
 (('things', 'which', 'are'), 66),
 (('say', ',', '"'), 65),
 ((',', 'and', 'a'), 64),
 (('.', 'Therefore', ','), 64)]

In [ ]:
def build_models(tokens):
    
    # buscando "iniciadores de frases"
    sentence_starters = [tokens[i+1] for i in range(len(tokens) - 1) if tokens[i] in ".!?"]
    start_counts = Counter(sentence_starters)
    total_starters = sum(start_counts.values())
    start_prob = {w: c / total_starters for w, c in start_counts.items()}
    
    # modelo bigrama para a segunda palavra frase
    bigram_counts = Counter(bigrams(tokens))
    bigram_model = defaultdict(lambda: defaultdict(float))
    for (w1, w2), c in bigram_counts.items():
        bigram_model[w1][w2] = c
    for w1 in bigram_model:
        total = sum(bigram_model[w1].values())
        for w2 in bigram_model[w1]:
            bigram_model[w1][w2] /= total
    
    # modelo trigrama aplicado a partir daqui
    trigram_counts = Counter(trigrams(tokens))
    trigram_model = defaultdict(lambda: defaultdict(float))
    for (w1, w2, w3), count in trigram_counts.items():
        trigram_model[(w1, w2)][w3] = count
    for w1_w2 in trigram_model:
        total = sum(trigram_model[w1_w2].values())
        for w3 in trigram_model[w1_w2]:
            trigram_model[w1_w2][w3] /= total  # Normalize

    return start_prob, bigram_model, trigram_model

In [ ]:
def generate_text(start_prob, bigram_model, trigram_model, length=15):
    
    w1 = random.choices(list(start_prob.keys()), weights=list(start_prob.values()))[0]

    if w1 in bigram_model and bigram_model[w1]:
        w2 = random.choices(list(bigram_model[w1].keys()), weights=list(bigram_model[w1].values()))[0]
    else:
        return w1  

    text = [w1, w2]  

    for _ in range(length - 2):
        if (w1, w2) not in trigram_model or not trigram_model[(w1, w2)]:
            break  # parar caso não haja palvra válida
        
        w3 = random.choices(
            list(trigram_model[(w1, w2)].keys()), 
            weights=list(trigram_model[(w1, w2)].values())
        )[0]
        
        text.append(w3)
        w1, w2 = w2, w3 

    return ' '.join(text)

In [17]:
start_prob, bigram_model, trigram_model = build_models(tokens)

generated_text = generate_text(start_prob, bigram_model, trigram_model)

In [43]:
generate_text(start_prob, bigram_model, trigram_model)

'Among the sounds that din round me , watch for all these arts of which'